<a href="https://colab.research.google.com/github/divyanshu05-codes/AI_Projects/blob/main/Dev1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
from sklearn.model_selection import train_test_split

nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

print("All libraries imported successfully!")
print("NLTK resources downloaded successfully!")
print(f"Number of English stopwords: {len(stop_words)}")

All libraries imported successfully!
NLTK resources downloaded successfully!
Number of English stopwords: 198


In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Once Google Drive is mounted successfully, update your file paths. If you uploaded the files directly to the root of your Google Drive, you can access them using the paths below:

In [11]:
# Load the CSV files directly from the current Colab workspace
train_df = pd.read_csv('/content/twitter_training.csv', names=columns)
val_df = pd.read_csv('/content/twitter_validation.csv', names=columns)

print("Data loaded successfully from the local workspace!")
print(f"Training set shape: {train_df.shape}")
print(f"Validation set shape: {val_df.shape}")

Data loaded successfully from the local workspace!
Training set shape: (74682, 4)
Validation set shape: (1000, 4)


In [12]:
train_df.head()

,tweet_id,Entity,Sentiment,Tweet
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...


In [13]:
val_df.head()

,tweet_id,Entity,Sentiment,Tweet
0,3364,Facebook,Irrelevant,I mentioned on Facebook that I was struggling ...
1,352,Amazon,Neutral,BBC News - Amazon boss Jeff Bezos rejects clai...
2,8312,Microsoft,Negative,@Microsoft Why do I pay for WORD when it funct...
3,4371,CS-GO,Negative,"CSGO matchmaking is so full of closet hacking,..."
4,4433,Google,Neutral,Now the President is slapping Americans in the...


In [14]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74682 entries, 0 to 74681
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   tweet_id   74682 non-null  int64 
 1   Entity     74682 non-null  object
 2   Sentiment  74682 non-null  object
 3   Tweet      73996 non-null  object
dtypes: int64(1), object(3)
memory usage: 2.3+ MB


In [15]:
train_df = train_df.dropna()

In [16]:
val_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   tweet_id   1000 non-null   int64 
 1   Entity     1000 non-null   object
 2   Sentiment  1000 non-null   object
 3   Tweet      1000 non-null   object
dtypes: int64(1), object(3)
memory usage: 31.4+ KB


In [17]:
train_df = train_df[['Sentiment','Tweet']]
val_df = val_df[['Sentiment','Tweet']]

In [18]:
train_df.duplicated().sum()

np.int64(4227)

In [19]:
train_df = train_df.drop_duplicates()

In [20]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 69769 entries, 0 to 74681
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Sentiment  69769 non-null  object
 1   Tweet      69769 non-null  object
dtypes: object(2)
memory usage: 1.6+ MB


In [21]:
val_df.duplicated().sum()

np.int64(1)

In [22]:
val_df= val_df.drop_duplicates()

In [23]:
val_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 999 entries, 0 to 999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Sentiment  999 non-null    object
 1   Tweet      999 non-null    object
dtypes: object(2)
memory usage: 23.4+ KB


In [25]:
train_df['Tweet'] = train_df['Tweet'].astype(str)
val_df['Tweet'] = val_df['Tweet'].astype(str)

In [26]:
val_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 999 entries, 0 to 999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Sentiment  999 non-null    object
 1   Tweet      999 non-null    object
dtypes: object(2)
memory usage: 23.4+ KB


In [27]:
train_df['Sentiment'].unique()

array(['Positive', 'Neutral', 'Negative', 'Irrelevant'], dtype=object)

In [28]:
stop_word = set(stopwords.words("english"))
stop_word = stop_word - {'not', 'no', 'nor', 'never'}

print('not' in stop_word)  # False

False


In [29]:
stop_word

{'a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 "he's",
 'her',
 'here',
 'hers',
 'herself',
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 "i'll",
 "i'm",
 "i've",
 'if',
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'only',
 'or',
 'other',
 

In [30]:
lemmatizer = WordNetLemmatizer()

In [31]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    clean_words = []
    for w in words:
        if w not in stop_word:
            word = lemmatizer.lemmatize(w)
            clean_words.append(word)
    return ' '.join(clean_words)

In [32]:
text = "Theo are good"
clean_text(text)

'theo good'

In [33]:
train_df['clean_tweet'] = train_df['Tweet'].apply(clean_text)
val_df['clean_tweet'] = val_df['Tweet'].apply(clean_text)

In [34]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 69769 entries, 0 to 74681
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Sentiment    69769 non-null  object
 1   Tweet        69769 non-null  object
 2   clean_tweet  69769 non-null  object
dtypes: object(3)
memory usage: 2.1+ MB


In [35]:
train_df.head()

,Sentiment,Tweet,clean_tweet
0,Positive,im getting on borderlands and i will murder yo...,im getting borderland murder
1,Positive,I am coming to the borders and I will kill you...,coming border kill
2,Positive,im getting on borderlands and i will kill you ...,im getting borderland kill
3,Positive,im coming on borderlands and i will murder you...,im coming borderland murder
4,Positive,im getting on borderlands 2 and i will murder ...,im getting borderland murder


In [36]:
val_df.head()

,Sentiment,Tweet,clean_tweet
0,Irrelevant,I mentioned on Facebook that I was struggling ...,mentioned facebook struggling motivation go ru...
1,Neutral,BBC News - Amazon boss Jeff Bezos rejects clai...,bbc news amazon bos jeff bezos reject claim co...
2,Negative,@Microsoft Why do I pay for WORD when it funct...,pay word function poorly chromebook
3,Negative,"CSGO matchmaking is so full of closet hacking,...",csgo matchmaking full closet hacking truly awf...
4,Neutral,Now the President is slapping Americans in the...,president slapping american face really commit...


In [37]:
train_df = train_df[train_df['clean_tweet'].str.strip() != '']
val_df = val_df[val_df['clean_tweet'].str.strip() != '']

In [38]:
val_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 999 entries, 0 to 999
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Sentiment    999 non-null    object
 1   Tweet        999 non-null    object
 2   clean_tweet  999 non-null    object
dtypes: object(3)
memory usage: 31.2+ KB


In [39]:
bow = CountVectorizer(max_features=5000)

x_train_bow = bow.fit_transform(train_df['clean_tweet'])
x_val_bow = bow.transform(val_df['clean_tweet'])

In [40]:
x_train_bow

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 622228 stored elements and shape (69204, 5000)>

In [41]:
tfidf = TfidfVectorizer(max_features=5000)

x_train_tfidf = tfidf.fit_transform(train_df['clean_tweet'])
x_val_tfidf = tfidf.transform(val_df['clean_tweet'])

In [42]:
x_train_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 622228 stored elements and shape (69204, 5000)>

In [43]:
y_train = train_df['Sentiment']
y_val = val_df['Sentiment']

In [44]:
model = LogisticRegression(max_iter=300)
model.fit(x_train_tfidf, y_train)

LogisticRegression(max_iter=300)

In [45]:
pred_df = model.predict(x_val_tfidf)

acc = accuracy_score(y_val, pred_df)
acc

0.7987987987987988

In [46]:
bow_model = LogisticRegression(max_iter=300)
bow_model.fit(x_train_bow, y_train)

bow_pred = bow_model.predict(x_val_bow)
bow_acc = accuracy_score(y_val, bow_pred)
print("BoW Accuracy:", bow_acc)

BoW Accuracy: 0.8408408408408409


In [48]:
new_tweet = ['pakistan has many terrorists','IPL 2K27 will come soon', 'BJP is the best']

clean_tweet=[]

for tweet in new_tweet:
    clean_tweet.append(clean_text(tweet))

new_features = tfidf.transform(clean_tweet)
predictions = bow_model.predict(new_features)

for tweet, sentiment in zip(new_tweet, predictions):
    print(tweet,'-->' ,sentiment)

pakistan has many terrorists --> Positive
IPL 2K27 will come soon --> Positive
BJP is the best --> Positive
